# DuckSAT Question Generation Workflow

This notebook provides a step-by-step process for generating SAT questions using AI.
Each cell represents a distinct step in the workflow, making it easy to track progress and identify errors.

## Overview
1. **Setup & Configuration** - Import libraries and set parameters
2. **Environment Check** - Verify required environment variables
3. **Server Connection Test** - Ensure the DuckSAT server is running
4. **Generate Questions** - Create AI-generated SAT questions
5. **Track Progress** - Monitor generation statistics
6. **View Results** - Display and analyze generated questions
7. **Error Handling** - Catch and display any errors that occur

## Step 1: Setup & Configuration

Import required libraries and set configuration parameters.

In [ ]:
# Import required libraries
import requests
import json
import time
import os
from datetime import datetime
from typing import Dict, List, Optional
from IPython.display import display, HTML, JSON
import sys

# Try to import dotenv for local .env file support
try:
    from dotenv import load_dotenv
    load_dotenv()
    print("✅ Loaded environment variables from .env file")
except ImportError:
    print("⚠️  python-dotenv not available, using system environment variables only")

print("✅ Libraries imported successfully")

## Step 2: Configuration Parameters

Set the parameters for question generation. Modify these as needed.

In [ ]:
# Configuration
config = {
    'base_url': os.getenv('BASE_URL', 'http://localhost:3000'),
    'admin_api_key': os.getenv('ADMIN_API_KEY', None),
    'question_count': int(os.getenv('QUESTION_COUNT', '10')),
    'batch_size': int(os.getenv('BATCH_SIZE', '5')),
    'batch_count': int(os.getenv('BATCH_COUNT', '1')),
    'delay_between_batches': int(os.getenv('DELAY_BETWEEN_BATCHES', '15000')) / 1000,  # Convert to seconds
    'module_type': os.getenv('MODULE_TYPE', None),  # 'math' or 'reading-writing'
    'difficulty': os.getenv('DIFFICULTY', None),  # 'easy', 'medium', or 'hard'
    'topic_id': os.getenv('TOPIC_ID', None),
    'subtopic_id': os.getenv('SUBTOPIC_ID', None),
    'temperature': float(os.getenv('TEMPERATURE', '0.7')),
    'max_tokens': int(os.getenv('MAX_TOKENS', '4000')),
    'include_charts': os.getenv('INCLUDE_CHARTS', 'true').lower() != 'false',
    'include_passages': os.getenv('INCLUDE_PASSAGES', 'true').lower() != 'false',
    'retry_attempts': int(os.getenv('RETRY_ATTEMPTS', '3')),
    'retry_delay': int(os.getenv('RETRY_DELAY', '5000')) / 1000,  # Convert to seconds
}

# Display configuration
print("📋 Configuration:")
print(f"   Base URL: {config['base_url']}")
print(f"   Admin API Key: {'***' + config['admin_api_key'][-4:] if config['admin_api_key'] else 'Not set (using session auth)'}")
print(f"   Question Count: {config['question_count']} per batch")
print(f"   Batch Size: {config['batch_size']} questions per request")
print(f"   Batch Count: {config['batch_count']} batches")
print(f"   Delay Between Batches: {config['delay_between_batches']}s")
print(f"   Module Type: {config['module_type'] or 'Both'}")
print(f"   Difficulty: {config['difficulty'] or 'All'}")
print(f"   Topic ID: {config['topic_id'] or 'Not specified'}")
print(f"   Subtopic ID: {config['subtopic_id'] or 'Not specified'}")
print(f"   Temperature: {config['temperature']}")
print(f"   Max Tokens: {config['max_tokens']}")
print(f"   Include Charts: {config['include_charts']}")
print(f"   Include Passages: {config['include_passages']}")
print(f"   Retry Attempts: {config['retry_attempts']}")
print("\n✅ Configuration loaded successfully")

## Step 3: Environment Validation

Validate configuration parameters.

In [ ]:
# Validation
errors = []

if config['question_count'] < 1 or config['question_count'] > 50:
    errors.append('QUESTION_COUNT must be between 1 and 50')

if config['batch_size'] < 1 or config['batch_size'] > 10:
    errors.append('BATCH_SIZE must be between 1 and 10')

if config['temperature'] < 0 or config['temperature'] > 2:
    errors.append('TEMPERATURE must be between 0 and 2')

if config['max_tokens'] < 1000 or config['max_tokens'] > 8000:
    errors.append('MAX_TOKENS must be between 1000 and 8000')

if config['module_type'] and config['module_type'] not in ['math', 'reading-writing']:
    errors.append('MODULE_TYPE must be "math" or "reading-writing"')

if config['difficulty'] and config['difficulty'] not in ['easy', 'medium', 'hard']:
    errors.append('DIFFICULTY must be "easy", "medium", or "hard"')

if errors:
    print("❌ Configuration errors:")
    for error in errors:
        print(f"   - {error}")
    raise ValueError("Configuration validation failed")
else:
    print("✅ Configuration validated successfully")

## Step 4: Server Connection Test

Test connection to the DuckSAT server before attempting generation.

In [ ]:
def test_connection(base_url: str, api_key: Optional[str] = None) -> bool:
    """Test server connection and authentication."""
    print("🔍 Testing server connection...")
    
    headers = {'Content-Type': 'application/json'}
    if api_key:
        headers['Authorization'] = f'Bearer {api_key}'
    
    try:
        response = requests.get(f"{base_url}/api/admin/questions", headers=headers, timeout=10)
        
        if response.status_code in [401, 403]:
            print("⚠️  Authentication required. Make sure you are logged in or have set ADMIN_API_KEY.")
            return False
        
        if not response.ok:
            print(f"⚠️  Server responded with status {response.status_code}")
            print(f"   Response: {response.text[:200]}")
            return False
        
        print("✅ Server is running and accessible")
        return True
        
    except requests.exceptions.ConnectionError:
        print(f"❌ Cannot connect to server at {base_url}")
        print("   Make sure the server is running with: npm run dev")
        return False
    except requests.exceptions.Timeout:
        print(f"❌ Connection timeout to {base_url}")
        return False
    except Exception as e:
        print(f"❌ Unexpected error: {str(e)}")
        return False

# Test the connection
connection_ok = test_connection(config['base_url'], config['admin_api_key'])

if not connection_ok:
    print("\n❌ Cannot proceed without server connection")
    print("   Please start the server and re-run this cell")
else:
    print("\n✅ Ready to generate questions")

## Step 5: Define Generation Functions

Helper functions for generating questions with retry logic.

In [ ]:
def format_duration(seconds: float) -> str:
    """Format duration in a human-readable way."""
    if seconds < 60:
        return f"{int(seconds)}s"
    minutes = int(seconds // 60)
    remaining_seconds = int(seconds % 60)
    if minutes < 60:
        return f"{minutes}m {remaining_seconds}s"
    hours = minutes // 60
    remaining_minutes = minutes % 60
    return f"{hours}h {remaining_minutes}m {remaining_seconds}s"

def generate_questions(config: dict, request_body: dict, attempt: int = 1) -> dict:
    """Generate questions with retry logic."""
    headers = {'Content-Type': 'application/json'}
    if config['admin_api_key']:
        headers['Authorization'] = f"Bearer {config['admin_api_key']}"
    
    try:
        response = requests.post(
            f"{config['base_url']}/api/admin/enhanced-generate-questions",
            headers=headers,
            json=request_body,
            timeout=300  # 5 minute timeout for generation
        )
        
        if not response.ok:
            error_text = response.text[:200]
            raise Exception(f"HTTP {response.status_code}: {error_text}")
        
        result = response.json()
        return {'success': True, 'data': result}
        
    except Exception as e:
        if attempt < config['retry_attempts']:
            print(f"   ⚠️  Attempt {attempt} failed: {str(e)}")
            print(f"   🔄 Retrying in {config['retry_delay']}s... (attempt {attempt + 1}/{config['retry_attempts']})")
            time.sleep(config['retry_delay'])
            return generate_questions(config, request_body, attempt + 1)
        
        return {'success': False, 'error': str(e)}

print("✅ Generation functions defined")

## Step 6: Initialize Statistics Tracking

Set up data structures to track generation progress.

In [ ]:
# Initialize statistics
stats = {
    'start_time': time.time(),
    'total_batches': 0,
    'successful_batches': 0,
    'failed_batches': 0,
    'total_generated': 0,
    'total_evaluated': 0,
    'total_accepted': 0,
    'total_rejected': 0,
    'total_stored': 0,
    'total_needs_review': 0,
    'errors': [],
    'batch_results': []
}

print("✅ Statistics tracking initialized")
print(f"   Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Step 7: Generate Questions - Main Loop

Execute the question generation process with progress tracking.

**This is the main generation step. It may take several minutes to complete.**

In [ ]:
print("🚀 Starting question generation...\n")

# Calculate question counts
total_questions = config['question_count']
if config['module_type'] == 'reading-writing':
    math_count = 0
    reading_count = total_questions
elif config['module_type'] == 'math':
    math_count = total_questions
    reading_count = 0
else:
    math_count = total_questions // 2
    reading_count = total_questions - math_count

print(f"📊 Generation Plan:")
print(f"   Total Questions: {total_questions} per batch")
print(f"   Math Questions: {math_count}")
print(f"   Reading Questions: {reading_count}")
print(f"   Number of Batches: {config['batch_count']}")
print()

# Run batches
for batch_num in range(1, config['batch_count'] + 1):
    batch_start = time.time()
    
    print("=" * 80)
    print(f"📦 Batch {batch_num}/{config['batch_count']}")
    print("=" * 80)
    print()
    
    stats['total_batches'] += 1
    
    # Build request body
    request_body = {
        'llmModel': 'gpt-5',
        'questionCount': total_questions,
        'mathCount': math_count,
        'readingCount': reading_count,
        'temperature': config['temperature'],
        'maxTokens': config['max_tokens'],
        'includeCharts': config['include_charts'],
        'includePassages': config['include_passages'],
    }
    
    # Add optional filters
    if config['topic_id']:
        request_body['topicId'] = config['topic_id']
    if config['subtopic_id']:
        request_body['subtopicId'] = config['subtopic_id']
    if config['module_type']:
        request_body['moduleType'] = config['module_type']
    if config['difficulty']:
        request_body['difficulty'] = config['difficulty']
    
    print(f"🔄 Generating {total_questions} questions (Math: {math_count}, Reading: {reading_count})...")
    
    # Generate questions
    result = generate_questions(config, request_body)
    
    batch_duration = time.time() - batch_start
    
    if result['success']:
        stats['successful_batches'] += 1
        data = result['data']
        
        if 'summary' in data:
            summary = data['summary']
            stats['total_generated'] += summary.get('generated', 0)
            stats['total_evaluated'] += summary.get('evaluated', 0)
            stats['total_accepted'] += summary.get('accepted', 0)
            stats['total_rejected'] += summary.get('rejected', 0)
            stats['total_stored'] += summary.get('stored', 0)
            stats['total_needs_review'] += summary.get('needsReview', 0)
            
            # Store batch result
            stats['batch_results'].append({
                'batch': batch_num,
                'success': True,
                'summary': summary,
                'duration': batch_duration
            })
            
            print("\n✅ Batch completed successfully!")
            print(f"   Generated: {summary.get('generated', 0)}")
            print(f"   Evaluated: {summary.get('evaluated', 0)}")
            print(f"   Accepted: {summary.get('accepted', 0)}")
            print(f"   Rejected: {summary.get('rejected', 0)}")
            print(f"   Stored: {summary.get('stored', 0)}")
            print(f"   Needs Review: {summary.get('needsReview', 0)}")
        
        print(f"   Duration: {format_duration(batch_duration)}")
        
    else:
        stats['failed_batches'] += 1
        error_msg = f"Batch {batch_num}: {result['error']}"
        stats['errors'].append(error_msg)
        
        # Store batch result
        stats['batch_results'].append({
            'batch': batch_num,
            'success': False,
            'error': result['error'],
            'duration': batch_duration
        })
        
        print(f"\n❌ Batch failed: {result['error']}")
    
    # Wait between batches (except after last batch)
    if batch_num < config['batch_count']:
        wait_seconds = config['delay_between_batches']
        print(f"\n⏳ Waiting {wait_seconds}s before next batch...")
        time.sleep(wait_seconds)
    
    print()

print("\n✅ Generation loop completed")

## Step 8: Display Final Summary

Show comprehensive statistics from the generation process.

In [ ]:
# Calculate final statistics
total_duration = time.time() - stats['start_time']

print("=" * 80)
print("🎉 BATCH GENERATION COMPLETE!")
print("=" * 80)
print()

print("📊 Final Statistics:")
print(f"   Total Batches: {stats['total_batches']}")
print(f"   Successful: {stats['successful_batches']} ✅")
print(f"   Failed: {stats['failed_batches']} ❌")
print()
print(f"   Questions Generated: {stats['total_generated']}")
print(f"   Questions Evaluated: {stats['total_evaluated']}")
print(f"   Questions Accepted: {stats['total_accepted']}")
print(f"   Questions Rejected: {stats['total_rejected']}")
print(f"   Questions Stored: {stats['total_stored']}")
print(f"   Questions Needing Review: {stats['total_needs_review']}")

if stats['total_generated'] > 0:
    acceptance_rate = (stats['total_accepted'] / stats['total_generated']) * 100
    print()
    print(f"   Acceptance Rate: {acceptance_rate:.1f}%")
    
    if stats['total_accepted'] > 0:
        storage_rate = (stats['total_stored'] / stats['total_accepted']) * 100
        print(f"   Storage Success Rate: {storage_rate:.1f}%")

print()
print(f"   Total Duration: {format_duration(total_duration)}")

if stats['total_stored'] > 0:
    avg_time_per_question = total_duration / stats['total_stored']
    print(f"   Average Time Per Stored Question: {format_duration(avg_time_per_question)}")

print()

## Step 9: Display Errors (if any)

Show any errors that occurred during generation.

In [ ]:
if stats['errors']:
    print("⚠️  Errors encountered:")
    for i, error in enumerate(stats['errors'], 1):
        print(f"   {i}. {error}")
else:
    print("✅ No errors encountered")

## Step 10: Display Warnings

Show any warnings about questions needing review.

In [ ]:
if stats['total_needs_review'] > 0:
    print("⚠️  Warning: Some questions need manual review!")
    print(f"   {stats['total_needs_review']} questions were flagged for review.")
    print(f"   Review them at: {config['base_url']}/admin/questions?reviewStatus=pending")
else:
    print("✅ No questions need review")

## Step 11: Display Batch Details

Show detailed results for each batch.

In [ ]:
print("\n📦 Batch Details:")
print()

for batch_result in stats['batch_results']:
    batch_num = batch_result['batch']
    print(f"Batch {batch_num}:")
    
    if batch_result['success']:
        summary = batch_result['summary']
        print(f"   ✅ Success")
        print(f"   Generated: {summary.get('generated', 0)}")
        print(f"   Accepted: {summary.get('accepted', 0)}")
        print(f"   Rejected: {summary.get('rejected', 0)}")
        print(f"   Stored: {summary.get('stored', 0)}")
    else:
        print(f"   ❌ Failed: {batch_result['error']}")
    
    print(f"   Duration: {format_duration(batch_result['duration'])}")
    print()

## Step 12: Visualize Results (Interactive)

Display an interactive JSON view of all batch results.

In [ ]:
# Create summary for JSON display
summary_data = {
    'configuration': {
        'base_url': config['base_url'],
        'question_count': config['question_count'],
        'batch_count': config['batch_count'],
        'module_type': config['module_type'] or 'both',
        'difficulty': config['difficulty'] or 'all',
    },
    'statistics': {
        'total_batches': stats['total_batches'],
        'successful_batches': stats['successful_batches'],
        'failed_batches': stats['failed_batches'],
        'total_generated': stats['total_generated'],
        'total_accepted': stats['total_accepted'],
        'total_rejected': stats['total_rejected'],
        'total_stored': stats['total_stored'],
        'total_needs_review': stats['total_needs_review'],
        'total_duration_seconds': round(total_duration, 2),
    },
    'batches': stats['batch_results'],
    'errors': stats['errors'],
}

display(JSON(summary_data, expanded=True))

## Step 13: Final Status

Display the final status of the generation process.

In [ ]:
if stats['failed_batches'] > 0:
    print("❌ Generation completed with errors")
    print(f"   {stats['failed_batches']} out of {stats['total_batches']} batches failed")
elif stats['total_needs_review'] > 0:
    print("⚠️  Generation completed successfully with warnings")
    print(f"   {stats['total_needs_review']} questions need manual review")
else:
    print("✅ Generation completed successfully!")
    print(f"   {stats['total_stored']} questions stored in database")

print()
print(f"View questions at: {config['base_url']}/admin/questions")

## Troubleshooting

If you encounter errors:

1. **Server Connection Failed**
   - Make sure the DuckSAT server is running: `npm run dev`
   - Check that `BASE_URL` is correct in your environment

2. **Authentication Errors**
   - Set `ADMIN_API_KEY` in your environment variables
   - Or ensure you're logged in to the web application

3. **Generation Timeout**
   - Reduce `QUESTION_COUNT` to generate fewer questions per batch
   - Increase timeout in the code if needed

4. **Rate Limiting**
   - Increase `DELAY_BETWEEN_BATCHES` to wait longer between batches
   - Reduce `BATCH_COUNT` to generate fewer batches

## Next Steps

- Review generated questions at `/admin/questions`
- Check questions flagged for review
- Adjust configuration and re-run if needed
- Export results for analysis